<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# MobileNetV1 on ImageNet - PyTorch to Akida 2

<p align="right">Run Time: ~5 minutes (sample pack) / ~1 hour (full ImageNet)</p>

This notebook takes a pretrained **PyTorch** ImageNet classifier, MobileNetV1, to
Akida 2, and checks the accuracy after every step:

1. Load the float PyTorch model
2. Export it to ONNX
3. Quantize it to 8 bits with **quantizeml** (post-training, no fine-tuning)
4. Convert it to an Akida model with **cnn2snn**

`quantizeml` reads the ONNX model directly, so the same route applies to other PyTorch
models whose layers Akida supports. The notebook ends with the
activation sparsity of the Akida model, which is what makes it cheap to run on Akida
hardware.

Steps 1 and 2 need PyTorch and timm, which are installed separately from the rest of
the repository: see Requirements in the README.

In [ ]:
# Colab-only setup. Local users: ignore this cell - it does nothing for you.
import sys
if 'google.colab' in sys.modules:
    !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida2/model_zoo/imagenet_mobilenet/colab_setup.py
    import colab_setup
    colab_setup.setup()

## Configuration

Pick a model by width multiplier: `ALPHA = 1.0` for `mobilenetv1_100` or `ALPHA = 1.25`
for `mobilenetv1_125`. Then pick the input resolution: `RESOLUTION = 224`, the resolution
the weights were trained at, or `RESOLUTION = 256`, the larger test resolution timm
publishes for the same weights. The README's Model Card explains why 256 scores higher.

Accuracy is measured on the 50,000-image ImageNet validation set, and quantization is
calibrated on images from the train split, if you have the dataset set up. ImageNet
cannot be redistributed, so that setup is manual - see 'Dataset' in the README. If the
dataset is not found, the notebook falls back to:

* a 10-image sample pack, fetched automatically, in place of the validation set. It is
  enough to exercise the whole pipeline, but far too small to measure accuracy with.
* the published quantized model from `pretrained_models/`, in place of quantizing, since
  calibration needs a few thousand real training images.

In [ ]:
import os
import numpy as np

ALPHA = 1.0          # 1.0 or 1.25
RESOLUTION = 224     # 224 or 256
BATCH_SIZE = 128
NUM_CALIB_SAMPLES = 8192

DATA_PATH = './data/imagenet'
MODELS_DIR = './models'   # models created by this notebook; the published ones are in pretrained_models/
os.makedirs(MODELS_DIR, exist_ok=True)

# Is the full dataset available?
USE_FULL_IMAGENET = all(os.path.isdir(os.path.join(DATA_PATH, split))
                        for split in ('train', 'validation'))

from imagenet_mobilenet_model import TIMM_NAMES, TIMM_REFERENCE
print(f'Model: {TIMM_NAMES[ALPHA]} (alpha={ALPHA}), {RESOLUTION} x {RESOLUTION} input')
print('Evaluating on:', 'full ImageNet validation set (50,000 images)'
      if USE_FULL_IMAGENET else '10-image sample pack (smoke test only)')

## 1. The float model

`timm.create_model(..., pretrained=True)` downloads the weights from the Hugging Face hub.

MobileNetV1 is a stack of depthwise-separable blocks: a 3 x 3 depthwise convolution
followed by a 1 x 1 pointwise convolution, each with BatchNorm and ReLU6. Every one of
those layers has an Akida equivalent, so the whole network maps to hardware. The width
multiplier (alpha) scales the number of channels in every layer.

In [ ]:
from imagenet_mobilenet_model import create_float_model

pt_model = create_float_model(ALPHA)
print(f'pretrained weights: {pt_model.pretrained_cfg["hf_hub_id"]}')
print(f'parameters:         {sum(p.numel() for p in pt_model.parameters()):,}')

## Data

Preprocessing is the evaluation transform timm resolves from the model's own
`pretrained_cfg`: a bicubic resize of the shorter side, then a centre crop to
`RESOLUTION x RESOLUTION`. The crop ratio depends on the checkpoint and the resolution:

| alpha | resolution | resize shorter side to | crop |
|---|---|---|---|
| 1.0 | 224 | 256 | 224 |
| 1.0 | 256 | 269 | 256 |
| 1.25 | 224 | 248 | 224 |
| 1.25 | 256 | 256 | 256 |

[imagenet_mobilenet_preprocessing.py](imagenet_mobilenet_preprocessing.py) writes it out
in full.

There are two flavours of the same images:

* **PyTorch and ONNX models** get float arrays, scaled to [0, 1] and normalised with mean
  0.5 and std 0.5, i.e. mapped to [-1, 1].
* **Akida models** get the raw uint8 pixels. `cnn2snn` folds the normalisation into the first
  layer, so normalising the data yourself as well would break the model.

Both are channels-first, (N, 3, RESOLUTION, RESOLUTION), as PyTorch exported them.

In [ ]:
from imagenet_mobilenet_data import get_data, get_labelled_samples
from imagenet_mobilenet_eval import evaluate_model, run_smoke_test


def evaluate(model, variant):
    """Top-1 and top-5 accuracy: over the validation set, or the sample pack."""
    if not USE_FULL_IMAGENET:
        return run_smoke_test(model, ALPHA, variant, RESOLUTION)
    loader, _ = get_data(DATA_PATH, ALPHA, batch_size=BATCH_SIZE,
                         normalize=variant != 'akida', resolution=RESOLUTION)
    top1, top5, _ = evaluate_model(model, loader)
    return top1, top5


float_images, _ = get_labelled_samples(ALPHA, normalize=True, resolution=RESOLUTION)
raw_images, _ = get_labelled_samples(ALPHA, normalize=False, resolution=RESOLUTION)
print(f'PyTorch/ONNX input: {float_images.shape} {float_images.dtype}, '
      f'range [{float_images.min():.2f}, {float_images.max():.2f}]')
print(f'Akida input:        {raw_images.shape} {raw_images.dtype}, '
      f'range [{raw_images.min()}, {raw_images.max()}]')

### Float evaluation

The baseline for everything that follows. On the full validation set this should match the
accuracy timm publishes for these weights.

In [ ]:
results = {}
results['float'] = evaluate(pt_model, 'float')
print(f'\nFloat: top-1 {results["float"][0] * 100:.2f}%, top-5 {results["float"][1] * 100:.2f}%')
ref_top1, ref_top5 = TIMM_REFERENCE[(ALPHA, RESOLUTION)]
print(f'timm reference: top-1 {ref_top1:.2f}%, top-5 {ref_top5:.2f}%')

## 2. Export to ONNX

`quantizeml` cannot read a PyTorch model, but it can read ONNX. The export traces the model
with a dummy input and keeps the batch axis dynamic. It is a change of format, not of the
computation, so the ONNX model should score exactly the same as the float one.

In [ ]:
from imagenet_mobilenet_model import export_onnx, model_path

model_onnx = export_onnx(pt_model, model_path(ALPHA, 'onnx', MODELS_DIR, RESOLUTION),
                         RESOLUTION)

results['onnx'] = evaluate(model_onnx, 'onnx')
print(f'\nONNX: top-1 {results["onnx"][0] * 100:.2f}%, top-5 {results["onnx"][1] * 100:.2f}%')

## 3. Quantize

Akida 2 runs 8-bit weights and activations. `quantizeml.models.quantize` converts the float
ONNX model to 8-bit and calibrates the activation ranges on sample images. Weights use
adaptive rounding (`weight_rounding="adaptive"`), which picks each weight's rounding
direction to minimise the layer's output error rather than rounding to nearest. The calibration images come from the **train** split, so the validation set stays
unseen. They are preprocessed exactly as for the float model.

The README's Model Card has the figures over the full validation set.

Without the dataset, we load the published quantized model instead.

In [ ]:
import onnx
from imagenet_mobilenet_data import get_calibration_samples
from imagenet_mobilenet_model import load_model, quantize_model

if USE_FULL_IMAGENET:
    calib_samples = get_calibration_samples(DATA_PATH, ALPHA, num_samples=NUM_CALIB_SAMPLES,
                                            batch_size=BATCH_SIZE, resolution=RESOLUTION)
    print(f'Calibration samples: {calib_samples.shape}')
    model_quantized = quantize_model(model_onnx, calib_samples, batch_size=BATCH_SIZE)
    onnx.save_model(model_quantized, str(model_path(ALPHA, 'quantized', MODELS_DIR, RESOLUTION)))
else:
    print('No ImageNet train split for calibration - loading the published quantized model.')
    model_quantized = load_model(ALPHA, 'quantized', resolution=RESOLUTION)

results['quantized'] = evaluate(model_quantized, 'quantized')
print(f'\nQuantized: top-1 {results["quantized"][0] * 100:.2f}%, '
      f'top-5 {results["quantized"][1] * 100:.2f}%')
print(f'Change vs float: {(results["quantized"][0] - results["float"][0]) * 100:+.2f} points top-1')

## 4. Convert to Akida

`cnn2snn.convert` turns the quantized ONNX model into an Akida model. It needs no data. If no
Akida hardware is present, the model runs on the software backend, which is what happens on
Colab.

The converted model should score about the same as the quantized one. The conversion
changes the representation, not the computation.

In [ ]:
from imagenet_mobilenet_model import convert_model

akida_model = convert_model(model_quantized)
akida_model.save(str(model_path(ALPHA, 'akida', MODELS_DIR, RESOLUTION)))
akida_model.summary()

In [ ]:
results['akida'] = evaluate(akida_model, 'akida')
print(f'\nAkida: top-1 {results["akida"][0] * 100:.2f}%, top-5 {results["akida"][1] * 100:.2f}%')

## Activation sparsity

Akida is event-driven: a zero activation generates no event, and therefore no work. The
higher the activation sparsity, the less the hardware actually has to do.

It is also why the hardware benchmark must use real images. Random noise gives the wrong
activity statistics, and so the wrong latency figures.

`compute_sparsity` lists every layer. In MobileNetV1 every convolution ends in a ReLU6,
and only the classifier has no activation. Its signed outputs are almost never exactly
zero, so the headline figure is the mean over the ReLU layers only.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from imagenet_mobilenet_data import get_samples
from imagenet_mobilenet_eval import mean_activation_sparsity

samples = get_samples(ALPHA, num_samples=100, data_path=DATA_PATH if USE_FULL_IMAGENET else None,
                      resolution=RESOLUTION)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)
mean_sparsity = mean_activation_sparsity(akida_model, sparsity_dict)
print(f'\nMean activation sparsity (ReLU layers): {mean_sparsity * 100:.2f}%')

## Summary

In [ ]:
print(f'{TIMM_NAMES[ALPHA]} (alpha={ALPHA}), {RESOLUTION} x {RESOLUTION}')
print(f'  measured on: {"50,000 validation images" if USE_FULL_IMAGENET else "10 sample images (smoke test)"}')
print()
print(f'  {"":12}{"top-1":>10}{"top-5":>10}')
print(f'  {"timm ref.":12}{ref_top1:9.2f}%{ref_top5:9.2f}%')
for variant, (top1, top5) in results.items():
    print(f'  {variant:12}{top1 * 100:9.2f}%{top5 * 100:9.2f}%')
print()
print(f'  mean activation sparsity (ReLU layers): {mean_sparsity * 100:.2f}%')

if not USE_FULL_IMAGENET:
    print('\nNOTE: these are 10-image smoke-test numbers, not accuracy measurements.')
    print('See the README Model Card for figures over the full validation set.')

To measure latency on Akida 2 hardware, continue with
[imagenet_mobilenet_notebook_benchmark.ipynb](imagenet_mobilenet_notebook_benchmark.ipynb).